In [1]:
"""
=============================================================
  PREDICCIÓN DE DEMANDA — XGBoost
  Comparación: 3 casos de limpieza x 2 frecuencias (mensual / semanal)
  Evaluación: Walk-Forward Cross-Validation (sin data leakage)
=============================================================

ESTRUCTURA ESPERADA DE ARCHIVOS:
  datos_mensuales/
      CafeBelenTotal.xlsx          ← hoja "Datos Producto"
  datos_semanales/
      Producto_281670015_semanal.csv
      Producto_105302070_semanal.csv
      ... (un CSV por producto)

INSTALACIÓN:
  pip install xgboost pandas openpyxl scikit-learn matplotlib seaborn
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import xgboost as xgb
    USE_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    USE_XGB = False
    print("⚠  xgboost no instalado — usando GradientBoostingRegressor de sklearn")

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# CONFIGURACIÓN — editá estos paths según tu estructura
# ─────────────────────────────────────────────────────────────
CONFIG = {
    # Archivo mensual
    "archivo_mensual": r"G:\My Drive\ElCapstone\Capstone\clari\CafeBelenTotal.xlsx",
    "hoja_mensual": "Datos Producto",

    # Carpeta con CSVs semanales (Producto_XXXXXXX_semanal.csv)
    "carpeta_semanal": r"datos_semanales/",

    # ID del producto a analizar (debe coincidir con los CSV semanales)
    # 281670015 = Café Belén
    "producto_id": 281670015,

    # Mínimo de períodos de historia antes de empezar a predecir (walk-forward)
    "min_train_mensual": 24,   # meses
    "min_train_semanal": 52,   # semanas (~1 año)

    # Carpeta de salida para gráficos
    "carpeta_output": "resultados/",
}

os.makedirs(CONFIG["carpeta_output"], exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1. CARGA DE DATOS
# ─────────────────────────────────────────────────────────────

def cargar_mensual(path, hoja, producto_id):
    df = pd.read_excel(path, sheet_name=hoja)
    df = df[df["Producto"] == producto_id].copy()
    df["Mes"] = pd.to_datetime(df["Mes"])
    df = df.sort_values("Mes")
    return df

def cargar_semanal(carpeta, producto_id):
    path = os.path.join(carpeta, f"Producto_{producto_id}_semanal.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"No encontré: {path}")
    df = pd.read_csv(path)
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df.sort_values("Fecha").reset_index(drop=True)
    return df


# ─────────────────────────────────────────────────────────────
# 2. PREPROCESSING — 3 CASOS
# ─────────────────────────────────────────────────────────────

def preprocesar_mensual(df_raw, caso):
    """
    Aplica el caso de limpieza a datos mensuales y agrega por mes.
    Retorna una Serie con índice datetime (frecuencia mensual).
    """
    d = df_raw.copy()

    if caso == 1:
        # Negativos → 0, ceros se mantienen
        d["Demanda"] = d["Demanda"].clip(lower=0)

    elif caso == 2:
        # Negativos → 0, eliminar ceros
        d["Demanda"] = d["Demanda"].clip(lower=0)
        d = d[d["Demanda"] > 0]

    elif caso == 3:
        # Netear devoluciones contra el mes anterior del mismo cliente
        d = d.sort_values(["NomClienteAlter", "Mes"])
        for idx in d[d["Demanda"] < 0].index:
            cliente = d.loc[idx, "NomClienteAlter"]
            mes_dev = d.loc[idx, "Mes"]
            devolucion = d.loc[idx, "Demanda"]
            prev = d[(d["NomClienteAlter"] == cliente) & (d["Mes"] < mes_dev)]
            if not prev.empty:
                prev_idx = prev.index[-1]
                d.loc[prev_idx, "Demanda"] = max(0, d.loc[prev_idx, "Demanda"] + devolucion)
            d.loc[idx, "Demanda"] = 0

    ts = d.groupby("Mes")["Demanda"].sum()
    ts = ts.reindex(pd.date_range(ts.index.min(), ts.index.max(), freq="MS"))
    ts = ts.fillna(ts.median())
    return ts


def preprocesar_semanal(df_raw, caso):
    """
    Aplica el caso de limpieza a datos semanales.
    Nota: los CSV ya tienen negativos→0 aplicados por el script de generación.
    El caso 3 semanal aplica lógica equivalente sobre la serie agregada.
    Retorna una Serie con índice datetime (frecuencia semanal).
    """
    d = df_raw.copy()
    d = d.set_index("Fecha")["Demanda"]

    if caso == 1:
        d = d.clip(lower=0)

    elif caso == 2:
        d = d.clip(lower=0)
        d = d[d > 0]
        d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq="W-MON")).fillna(0)
        # Re-eliminar ceros introducidos por reindex
        d = d[d > 0]

    elif caso == 3:
        # En datos semanales ya neteados, aplicamos suavizado de outliers bajos
        # (reemplazamos ceros aislados con media de vecinos)
        d = d.clip(lower=0)
        vals = d.values.copy()
        for i in range(1, len(vals) - 1):
            if vals[i] == 0 and vals[i-1] > 0 and vals[i+1] > 0:
                vals[i] = (vals[i-1] + vals[i+1]) / 2
        d = pd.Series(vals, index=d.index)

    return d


# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────

def construir_features(ts, freq="mensual"):
    """
    Construye DataFrame con features temporales y lags.
    freq: 'mensual' o 'semanal'
    """
    df = pd.DataFrame({"y": ts}).reset_index()
    df.columns = ["fecha", "y"]
    df = df.sort_values("fecha").reset_index(drop=True)

    # Features de calendario
    df["mes_num"]    = df["fecha"].dt.month
    df["trimestre"]  = df["fecha"].dt.quarter
    df["año"]        = df["fecha"].dt.year
    df["t"]          = range(len(df))                      # tendencia lineal

    if freq == "semanal":
        df["semana_año"] = df["fecha"].dt.isocalendar().week.astype(int)
        df["dia_año"]    = df["fecha"].dt.dayofyear

    # Lags
    lags = [1, 2, 3, 4, 6, 12] if freq == "mensual" else [1, 2, 3, 4, 8, 13, 26, 52]
    for lag in lags:
        df[f"lag_{lag}"] = df["y"].shift(lag)

    # Rolling stats (sobre lag-1 para no filtrar el futuro)
    ventanas = [3, 6] if freq == "mensual" else [4, 8, 13, 26]
    for v in ventanas:
        df[f"roll_mean_{v}"] = df["y"].shift(1).rolling(v).mean()
        df[f"roll_std_{v}"]  = df["y"].shift(1).rolling(v).std()
        df[f"roll_max_{v}"]  = df["y"].shift(1).rolling(v).max()

    # Diferencia con período anterior
    df["diff_1"] = df["y"].diff(1)

    return df.dropna().reset_index(drop=True)


# ─────────────────────────────────────────────────────────────
# 4. MODELO
# ─────────────────────────────────────────────────────────────

def construir_modelo():
    if USE_XGB:
        return xgb.XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            verbosity=0,
        )
    else:
        return GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=42,
        )


# ─────────────────────────────────────────────────────────────
# 5. WALK-FORWARD CROSS-VALIDATION
# ─────────────────────────────────────────────────────────────

def walk_forward_cv(ts, min_train, freq="mensual"):
    """
    Entrena el modelo con todos los datos hasta t-1,
    predice t, avanza un paso. Nunca ve el futuro.
    Retorna listas: actuals, preds, fechas.
    """
    df = construir_features(ts, freq)
    feature_cols = [c for c in df.columns if c not in ["fecha", "y"]]

    X = df[feature_cols].values
    y = df["y"].values
    fechas = df["fecha"].values

    actuals, preds, dates = [], [], []

    for i in range(min_train, len(df)):
        X_train, y_train = X[:i], y[:i]
        X_test = X[i : i + 1]

        model = construir_modelo()
        model.fit(X_train, y_train)
        pred = float(model.predict(X_test)[0])
        pred = max(0, pred)   # demanda no puede ser negativa

        preds.append(pred)
        actuals.append(float(y[i]))
        dates.append(pd.Timestamp(fechas[i]))

    return np.array(actuals), np.array(preds), dates


# ─────────────────────────────────────────────────────────────
# 6. MÉTRICAS
# ─────────────────────────────────────────────────────────────

def calcular_metricas(actuals, preds, label=""):
    """
    MAE     : error absoluto medio (unidades)
    RMSE    : raíz del error cuadrático medio (penaliza errores grandes)
    MAPE    : % error (solo donde actuals > 0, sensible a valores bajos)
    SMAPE   : % error simétrico (más robusto, usa promedio actuals+preds)
    Bias    : sesgo sistemático (+ = sobreestima, - = subestima)
    R²      : varianza explicada (1.0 = perfecto, <0 = peor que media)
    """
    mae  = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))

    mask = actuals > 0
    mape = np.mean(np.abs((actuals[mask] - preds[mask]) / actuals[mask])) * 100

    denom = (np.abs(actuals) + np.abs(preds) + 1e-9)
    smape = np.mean(2 * np.abs(actuals - preds) / denom) * 100

    bias = np.mean(preds - actuals)

    ss_res = np.sum((actuals - preds) ** 2)
    ss_tot = np.sum((actuals - actuals.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    metricas = {
        "label": label,
        "n": len(actuals),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "SMAPE": smape,
        "Bias": bias,
        "R2": r2,
    }
    return metricas


def imprimir_metricas(m):
    print(f"\n{'─'*55}")
    print(f"  {m['label']}  (n={m['n']} puntos)")
    print(f"{'─'*55}")
    print(f"  MAE   : {m['MAE']:>8.1f}  unidades por período")
    print(f"  RMSE  : {m['RMSE']:>8.1f}  penaliza errores grandes")
    print(f"  MAPE  : {m['MAPE']:>7.2f}%  error porcentual")
    print(f"  SMAPE : {m['SMAPE']:>7.2f}%  más robusto que MAPE")
    print(f"  Bias  : {m['Bias']:>+8.1f}  (+sobreestima / -subestima)")
    print(f"  R²    : {m['R2']:>8.3f}  varianza explicada")


# ─────────────────────────────────────────────────────────────
# 7. VISUALIZACIONES
# ─────────────────────────────────────────────────────────────

COLORES_CASO = {
    "Caso 1": "#378ADD",
    "Caso 2": "#D85A30",
    "Caso 3": "#1D9E75",
}

def grafico_predicciones(resultados, freq, producto_id, carpeta):
    """Un subplot por caso: real vs predicho."""
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)
    fig.suptitle(
        f"Predicciones vs Real — {freq.capitalize()} | Producto {producto_id}",
        fontsize=14, fontweight="bold", y=0.98
    )

    for ax, (key, r) in zip(axes, resultados.items()):
        color = list(COLORES_CASO.values())[list(resultados.keys()).index(key)]
        ax.plot(r["dates"], r["actuals"], label="Real", color="#888780", linewidth=1.5, alpha=0.9)
        ax.plot(r["dates"], r["preds"],   label="Predicho", color=color,
                linewidth=1.5, linestyle="--", alpha=0.9)
        ax.fill_between(r["dates"], r["actuals"], r["preds"],
                        alpha=0.08, color=color)
        ax.set_title(
            f"{key}  |  SMAPE {r['metricas']['SMAPE']:.1f}%  |  Bias {r['metricas']['Bias']:+.0f}",
            fontsize=11
        )
        ax.set_ylabel("Demanda")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    path = os.path.join(carpeta, f"predicciones_{freq}_{producto_id}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Guardado: {path}")


def grafico_comparativa(resultados_mensual, resultados_semanal, producto_id, carpeta):
    """Comparativa de métricas entre todos los casos y frecuencias."""
    rows = []
    for key, r in resultados_mensual.items():
        m = r["metricas"]
        rows.append({"Caso": key, "Freq": "Mensual",
                     "SMAPE": m["SMAPE"], "MAE": m["MAE"],
                     "RMSE": m["RMSE"], "Bias": m["Bias"], "R2": m["R2"]})
    for key, r in resultados_semanal.items():
        m = r["metricas"]
        rows.append({"Caso": key, "Freq": "Semanal",
                     "SMAPE": m["SMAPE"], "MAE": m["MAE"],
                     "RMSE": m["RMSE"], "Bias": m["Bias"], "R2": m["R2"]})

    df_res = pd.DataFrame(rows)
    df_res["Etiqueta"] = df_res["Caso"] + "\n" + df_res["Freq"]

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"Comparativa completa — Producto {producto_id}", fontsize=14, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    metricas_plot = [
        ("SMAPE",  "SMAPE (%)",           gs[0, 0]),
        ("MAE",    "MAE (unidades)",       gs[0, 1]),
        ("RMSE",   "RMSE (unidades)",      gs[0, 2]),
        ("Bias",   "Bias (unidades)",      gs[1, 0]),
        ("R2",     "R² (varianza expl.)",  gs[1, 1]),
    ]

    paleta = {"Mensual": "#85B7EB", "Semanal": "#1D9E75"}

    for metrica, ylabel, pos in metricas_plot:
        ax = fig.add_subplot(pos)
        vals  = df_res[metrica].values
        etiq  = df_res["Etiqueta"].values
        freqs = df_res["Freq"].values
        colores = [paleta[f] for f in freqs]

        bars = ax.bar(etiq, vals, color=colores, width=0.5, edgecolor="none")
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + abs(max(vals) - min(vals)) * 0.02,
                f"{val:.1f}" if metrica != "R2" else f"{val:.3f}",
                ha="center", va="bottom", fontsize=8
            )
        ax.set_title(ylabel, fontsize=10)
        ax.tick_params(axis="x", labelsize=8)
        ax.grid(axis="y", alpha=0.3)

        if metrica == "Bias":
            ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")

    # Leyenda
    from matplotlib.patches import Patch
    ax_legend = fig.add_subplot(gs[1, 2])
    ax_legend.axis("off")
    legend_elements = [
        Patch(facecolor="#85B7EB", label="Mensual"),
        Patch(facecolor="#1D9E75", label="Semanal"),
    ]
    ax_legend.legend(handles=legend_elements, loc="center", fontsize=11,
                     title="Frecuencia", title_fontsize=11)

    path = os.path.join(carpeta, f"comparativa_{producto_id}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Guardado: {path}")
    return df_res


def grafico_heatmap(df_res, producto_id, carpeta):
    """Heatmap de SMAPE para ver de un vistazo qué combinación gana."""
    pivot = df_res.pivot(index="Caso", columns="Freq", values="SMAPE")

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
        linewidths=0.5, ax=ax, cbar_kws={"label": "SMAPE (%)"}
    )
    ax.set_title(f"SMAPE (%) por caso y frecuencia\nProducto {producto_id}", fontsize=11)
    plt.tight_layout()
    path = os.path.join(carpeta, f"heatmap_smape_{producto_id}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Guardado: {path}")


def tabla_resumen(df_res, producto_id, carpeta):
    """Guarda tabla de resultados como CSV y la imprime."""
    path = os.path.join(carpeta, f"resumen_metricas_{producto_id}.csv")
    df_res.round(3).to_csv(path, index=False)
    print(f"\n{'═'*65}")
    print(f"  TABLA RESUMEN — Producto {producto_id}")
    print(f"{'═'*65}")
    print(df_res[["Etiqueta", "SMAPE", "MAE", "RMSE", "Bias", "R2"]]
          .sort_values("SMAPE")
          .to_string(index=False))
    print(f"{'═'*65}")
    print(f"\n  Mejor combinación: {df_res.loc[df_res['SMAPE'].idxmin(), 'Etiqueta'].replace(chr(10), ' ')}")
    print(f"  → Guardado: {path}")


# ─────────────────────────────────────────────────────────────
# 8. PIPELINE PRINCIPAL
# ─────────────────────────────────────────────────────────────

def correr_frecuencia(ts, min_train, freq, casos_labels):
    """Corre walk-forward CV para los 3 casos sobre una serie dada."""
    resultados = {}
    for caso_num, label in casos_labels.items():
        print(f"  Entrenando {label} ({freq}) ...")
        actuals, preds, dates = walk_forward_cv(ts, min_train, freq)
        metricas = calcular_metricas(actuals, preds, label=f"{label} [{freq}]")
        imprimir_metricas(metricas)
        resultados[label] = {
            "actuals":  actuals,
            "preds":    preds,
            "dates":    dates,
            "metricas": metricas,
        }
    return resultados


def main():
    producto_id  = CONFIG["producto_id"]
    carpeta_out  = CONFIG["carpeta_output"]

    casos_labels = {
        1: "Caso 1",
        2: "Caso 2",
        3: "Caso 3",
    }

    # ── MENSUAL ──────────────────────────────────────────────
    print("\n" + "═"*65)
    print("  FRECUENCIA MENSUAL")
    print("═"*65)

    df_mensual_raw = cargar_mensual(
        CONFIG["archivo_mensual"],
        CONFIG["hoja_mensual"],
        producto_id,
    )

    resultados_mensual = {}
    for caso_num, label in casos_labels.items():
        ts = preprocesar_mensual(df_mensual_raw, caso_num)
        print(f"\n  {label} — serie: {len(ts)} meses | neg→0 | zeros: {(ts == 0).sum()}")
        actuals, preds, dates = walk_forward_cv(
            ts, CONFIG["min_train_mensual"], "mensual"
        )
        metricas = calcular_metricas(actuals, preds, f"{label} [Mensual]")
        imprimir_metricas(metricas)
        resultados_mensual[label] = {"actuals": actuals, "preds": preds,
                                     "dates": dates, "metricas": metricas}

    grafico_predicciones(resultados_mensual, "mensual", producto_id, carpeta_out)

    # ── SEMANAL ───────────────────────────────────────────────
    print("\n" + "═"*65)
    print("  FRECUENCIA SEMANAL")
    print("═"*65)

    df_semanal_raw = cargar_semanal(CONFIG["carpeta_semanal"], producto_id)

    resultados_semanal = {}
    for caso_num, label in casos_labels.items():
        ts = preprocesar_semanal(df_semanal_raw, caso_num)
        print(f"\n  {label} — serie: {len(ts)} semanas | zeros: {(ts == 0).sum()}")
        actuals, preds, dates = walk_forward_cv(
            ts, CONFIG["min_train_semanal"], "semanal"
        )
        metricas = calcular_metricas(actuals, preds, f"{label} [Semanal]")
        imprimir_metricas(metricas)
        resultados_semanal[label] = {"actuals": actuals, "preds": preds,
                                     "dates": dates, "metricas": metricas}

    grafico_predicciones(resultados_semanal, "semanal", producto_id, carpeta_out)

    # ── COMPARATIVA FINAL ─────────────────────────────────────
    print("\n" + "═"*65)
    print("  COMPARATIVA FINAL")
    print("═"*65)

    df_res = grafico_comparativa(resultados_mensual, resultados_semanal, producto_id, carpeta_out)
    grafico_heatmap(df_res, producto_id, carpeta_out)
    tabla_resumen(df_res, producto_id, carpeta_out)

    print("\n✅  Análisis completo. Revisá la carpeta:", os.path.abspath(carpeta_out))


if __name__ == "__main__":
    main()

⚠  xgboost no instalado — usando GradientBoostingRegressor de sklearn

═════════════════════════════════════════════════════════════════
  FRECUENCIA MENSUAL
═════════════════════════════════════════════════════════════════

  Caso 1 — serie: 62 meses | neg→0 | zeros: 1

───────────────────────────────────────────────────────
  Caso 1 [Mensual]  (n=26 puntos)
───────────────────────────────────────────────────────
  MAE   :    983.5  unidades por período
  RMSE  :   1185.2  penaliza errores grandes
  MAPE  :   30.68%  error porcentual
  SMAPE :   28.36%  más robusto que MAPE
  Bias  :   -127.7  (+sobreestima / -subestima)
  R²    :    0.607  varianza explicada

  Caso 2 — serie: 62 meses | neg→0 | zeros: 0

───────────────────────────────────────────────────────
  Caso 2 [Mensual]  (n=26 puntos)
───────────────────────────────────────────────────────
  MAE   :   1011.7  unidades por período
  RMSE  :   1198.0  penaliza errores grandes
  MAPE  :   29.86%  error porcentual
  SMAPE :   28

FileNotFoundError: No encontré: datos_semanales/Producto_281670015_semanal.csv